# 📅 Exercício Extra 2: Detecão de Sinais por Formas Geométricas 🛑🔺

As câmaras autónomas podem reconhecer sinais de trânsito identificando a sua forma geométrica. Um sinal de **STOP** é sempre um Octógono (8 lados) e um sinal de **Perigo/Prioridade** é muitas vezes um Triângulo (3 lados).

Vamos usar a função `cv2.approxPolyDP` do OpenCV, que simplifica um contorno e conta o número de vértices (cantos) que ele tem!

### 🎯 O Teu Objetivo
Mostrar um triângulo ou um octógono de papel à câmara do robô e fazer o código reconhecer a forma automaticamente.

### 🛠️ Instruções
1. Recorta um triângulo e um sinal de STOP em papel (podes pintá-los com cores vivas).
2. Executa a célula abaixo.
3. Mostra as formas à câmara e repara no texto que aparece por cima do sinal detetado!

In [ ]:
import cv2
import numpy as np
import ipywidgets as widgets
from IPython.display import display
from jetcam.csi_camera import CSICamera
import time

print("--- DETETOR GEOMÉTRICO DE SINAIS ATIVO ---")

camera = CSICamera(width=300, height=300, capture_width=1280, capture_height=720, capture_fps=15)
imagem_widget = widgets.Image(format='jpeg', width=300, height=300)
botao_desligar = widgets.Button(description="❌ DESLIGAR", button_style='danger')

display(imagem_widget, botao_desligar)
sistema_ativo = True

def detetar_formas(change):
    global sistema_ativo
    if not sistema_ativo: return
    
    frame = change['new']
    cinzento = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    # Aplica um desfoque para limpar o ruído da imagem
    desfocado = cv2.GaussianBlur(cinzento, (5, 5), 0)
    # Detetor de收 edges (Canny)
    bordas = cv2.Canny(desfocado, 50, 150)
    
    # Encontrar os contornos das formas
    contornos, _ = cv2.findContours(bordas, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    for c in contornos:
        if cv2.contourArea(c) < 1000: # Ignora contornos muito pequenos
            continue
            
        # Aproxima a forma geométrica
        perimetro = cv2.arcLength(c, True)
        aproximacao = cv2.approxPolyDP(c, 0.04 * perimetro, True)
        
        # CONTA OS VÉRTICES (LADOS)
        lados = len(aproximacao)
        
        # Coordenadas para desenhar o texto no ecrã
        x, y, w, h = cv2.boundingRect(aproximacao)
        
        # --- LÓGICA DE RECONHECIMENTO ---
        if lados == 3:
            cv2.putText(frame, "TRIANGULO (Aviso)", (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
            cv2.rectangle(frame, (x,y), (x+w, y+h), (0, 255, 0), 2)
        elif lados == 8:
            cv2.putText(frame, "STOP! (8 lados)", (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)
            cv2.rectangle(frame, (x,y), (x+w, y+h), (0, 0, 255), 2)
            
    _, jpeg = cv2.imencode('.jpg', frame)
    imagem_widget.value = jpeg.tobytes()
    time.sleep(0.02)

camera.observe(detetar_formas, names='value')

def encerra(b):
    global sistema_ativo
    sistema_ativo = False
    camera.unobserve(detetar_formas, names='value')
    camera.running = False
    print("Sistema desligado.")

botao_desligar.on_click(encerra)
camera.running = True